# TRIAGE on Kaggle T4×2 (free tier)

Full local pipeline, zero LLM API keys: generator `llama3.1:8b` + judge `qwen2.5:14b`, both via ollama.

**Before running: notebook Settings →**
- Accelerator: **GPU T4 x2**
- Internet: **ON** (requires phone-verified Kaggle account)

Session budget: **9 h per GPU commit run** (Save & Run All), ~29 GB system RAM. `prepare --rebuild` is the long pole (~75 min on 4 vCPU + T4 embedding). The benchmark checkpoints per question, so an interrupted run resumes safely. Plan: prepare + 24-question pilot in session 1; full benchmark in session 2 from restored artifacts.

**Judge options (edit `JUDGE_OLLAMA_MODEL` in cell 4):**
- `qwen2.5:14b` (default, recommended): 8b + 14b ≈ 13.7 GB — both fit one T4, no model swapping. Decent judging quality.
- `qwen2.5:7b`: faster pilot-grade fallback if VRAM gets tight.
- `qwen2.5:32b`: does NOT fit a 16 GB T4 (≈19 GB Q4); ollama CPU-offloads and judging crawls (~3–6 tok/s). Not recommended on this hardware — use the A100/Colab Pro or the `JUDGE_PROVIDER=huggingface` cloud hybrid for 32B+ quality.

In [1]:
# 1. Clone the repo (idempotent: clone once, then require a successful update)
import sys, subprocess, os, json
print(sys.version)
if sys.version_info >= (3, 13):
    print('Python 3.13 session detected - cell 2 will build a Python 3.12 side-env')

def run_checked(args, label):
    print(f'Running: {label}')
    return subprocess.run(args, check=True)

if not os.path.isdir('/kaggle/working/TRIAGE/.git'):
    if os.path.exists('/kaggle/working/TRIAGE'):
        import shutil
        shutil.rmtree('/kaggle/working/TRIAGE')
    run_checked(['git', 'clone', '--depth', '1', 'https://github.com/Nirajd071/TRIAGE.git', '/kaggle/working/TRIAGE'],
                'clone TRIAGE')
%cd /kaggle/working/TRIAGE
run_checked(['git', 'pull', '--rebase'], 'update repository')
run_checked(['ls', 'src'], 'verify source tree')
_head_result = subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                              capture_output=True, text=True, check=True)
_head = _head_result.stdout.strip()
print('repo HEAD:', _head)
_stale = subprocess.run(
    ['grep', '-q', '"documents", "metadatas", "ids"',
     'src/evaluation/run_base_paper_benchmark.py'], check=False)
if _stale.returncode == 0:
    raise RuntimeError(
        'STALE CLONE: the chromadb ids include bug is still present. '
        'Delete /kaggle/working/TRIAGE and clone the current main branch.'
    )
# Env-pin delivery check: the Oct 25 scispacy failure came from a
# dependency the fresh env build pulled in; the pins must be IN the clone
# or the run is doomed to repeat it.
_pin = subprocess.run(
    ['grep', '-q', 'murmurhash==1.0.12', 'requirements.txt'], check=False)
if _pin.returncode != 0:
    raise RuntimeError(
        'Clone missing the env pins (murmurhash==1.0.12 in '
        'requirements.txt). Delete /kaggle/working/TRIAGE and re-clone '
        'the current main branch.'
    )
print('Env pins present - delivery guard passed.')


3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Python 3.13 session detected - cell 2 will build a Python 3.12 side-env
Running: clone TRIAGE


Cloning into '/kaggle/working/TRIAGE'...


/kaggle/working/TRIAGE
Running: update repository
Already up to date.
Running: verify source tree
api
baselines
config.py
core
data
evaluation
__init__.py
main.py
wandb_tracking.py
repo HEAD: fd91810 Eval: Calibrate GO/STOP gate to recognize positive ranking improvement (dNDCG >= 0)
Env pins present - delivery guard passed.


In [ ]:
# 2. Install dependencies (10-20 min)
# Python 3.13 sessions build a Python 3.12 side-env; all later stages use PY.
import sys
PY = sys.executable
if sys.version_info >= (3, 13):
    run_checked(['bash', '-lc',
                 'set -o pipefail; curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest '
                 '| tar -xj -C /usr/local bin/micromamba'], 'install micromamba')
    run_checked(['micromamba', 'create', '-y', '-p', '/kaggle/tmp/py312', 'python=3.12'],
                'create Python 3.12 side-env')
    PY = '/kaggle/tmp/py312/bin/python'
run_checked([PY, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
            'install pinned project dependencies')
run_checked([PY, '-m', 'pip', 'install', '-q', 'murmurhash==1.0.12'],
            'pin murmurhash (tokenizer specials regression guard)')
run_checked([PY, '-m', 'pip', 'install', '-q', 'numpy==1.26.4'],
            'install SciSpacy-compatible NumPy')
run_checked([PY, '-m', 'pip', 'install', '-q', '--no-deps',
             'https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz'],
            'install BC5CDR model')
run_checked([PY, '-m', 'pip', 'install', '-q', '--no-deps',
             'https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_jnlpba_md-0.5.4.tar.gz'],
            'install JNLPBA model')
print('python:', PY)
run_checked([PY, '-c',
             'import chromadb, spacy, scispacy; '
             'print("stack OK:", chromadb.__version__, spacy.__version__, scispacy.__version__); '
             'from importlib.metadata import version as _v; import numpy; '
             'print("low-level:", _v("murmurhash"), numpy.__version__); '
             'assert chromadb.__version__ == "1.5.9"; '
             'assert spacy.__version__ == "3.8.11"; '
             'assert tuple(int(x) for x in numpy.__version__.split(".")[:2]) < (2, 0)'],
            'verify target Chroma and spaCy versions')


In [3]:
# 2b. Verify the CUDA runtime used by embedding and tensor training.
import shutil
if not shutil.which("nvidia-smi"):
    raise RuntimeError(
        "No GPU detected in this Kaggle session!\n"
        "Please enable a GPU accelerator in Kaggle Settings:\n"
        "  1. In the right-hand sidebar, click on Notebook options / Settings.\n"
        "  2. Under Accelerator, change from None (CPU) to GPU T4 x 2 (recommended) or GPU P100.\n"
        "  3. Restart the session and re-run."
    )
run_checked(["nvidia-smi"], "verify Kaggle GPUs")
_cuda_check = subprocess.run([PY, "-c", "import torch; print(torch.cuda.is_available())"],
                             capture_output=True, text=True, check=True)
cuda_ok = _cuda_check.stdout.strip().endswith("True")
print("torch.cuda.is_available():", cuda_ok)
if not cuda_ok:
    print("CUDA unavailable - reinstalling torch with CUDA 12.6 runtime...")
    run_checked([PY, "-m", "pip", "install", "-q", "--force-reinstall", "torch",
                 "--index-url", "https://download.pytorch.org/whl/cu126"],
                "install CUDA 12.6 PyTorch")
    _cuda_check = subprocess.run(
        [PY, "-c", "import torch; print(torch.cuda.is_available()); "
         "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"none\")"],
        capture_output=True, text=True, check=True)
    print("after fix:", _cuda_check.stdout.strip())
    if not _cuda_check.stdout.splitlines()[0].strip().endswith("True"):
        raise RuntimeError("CUDA remains unavailable after the PyTorch reinstall.")
else:
    run_checked([PY, "-c", "import torch; print(\"GPU:\", torch.cuda.get_device_name(0))"],
                "verify CUDA device")


Running: verify Kaggle GPUs
Wed Oct 28 06:29:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------

In [4]:
# 3. Install ollama, start it on GPU 1 (or 0 if single-GPU), and verify both pinned models.
run_checked(['apt-get', '-qq', 'install', '-y', 'zstd'], 'install zstd')
run_checked(['bash', '-lc', 'set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh'],
            'install ollama')
import os, time, subprocess
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'
_gpu_res = subprocess.run(['nvidia-smi', '--query-gpu=count', '--format=csv,noheader'], capture_output=True, text=True)
_gpu_count = int(_gpu_res.stdout.strip().splitlines()[0]) if _gpu_res.returncode == 0 and _gpu_res.stdout.strip() else 1
_ollama_gpu = '1' if _gpu_count > 1 else '0'
print(f'Detected {_gpu_count} GPU(s); assigning Ollama to GPU {_ollama_gpu}')
get_ipython().system_raw(f'CUDA_VISIBLE_DEVICES={_ollama_gpu} ollama serve > /tmp/ollama.log 2>&1 &')
time.sleep(5)
run_checked(['ollama', 'pull', 'llama3.1:8b'], 'pull generator model')
run_checked(['ollama', 'pull', 'qwen2.5:14b'], 'pull judge model')
run_checked(['ollama', 'list'], 'verify ollama models')


Running: install zstd
Selecting previously unselected package zstd.
(Reading database ... 129796 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
Running: install ollama


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Detected 2 GPU(s); assigning Ollama to GPU 1
Running: pull generator model


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏ 2.7 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   1% ▕                  ▏  31 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   2% ▕                  ▏ 113 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   4% ▕                  ▏ 209 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   5% ▕                  ▏ 250 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   7% ▕█                 ▏ 347 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   9% ▕█                 ▏ 445 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:  10% ▕█                 ▏ 490 MB/4.9 GB                  pulling manifes

Running: pull judge model


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 2049f5674b1e:   0% ▕                  ▏  41 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   1% ▕                  ▏  81 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   2% ▕                  ▏ 174 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   3% ▕                  ▏ 262 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   3% ▕                  ▏ 306 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   4% ▕                  ▏ 402 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   5% ▕                  ▏ 432 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   6% ▕█                 ▏ 524 MB/9.0 GB                  pulling manifest 
pulling 2049f567

Running: verify ollama models
NAME           ID              SIZE      MODIFIED               
qwen2.5:14b    7cdf5a0187d5    9.0 GB    Less than a second ago    
llama3.1:8b    46e0c10c039e    4.9 GB    About a minute ago        


CompletedProcess(args=['ollama', 'list'], returncode=0)

In [5]:
# 4. Environment file - all API keys go HERE (this is the only cell with keys).
#    Preferred: Notebook editor -> Add-ons -> Secrets -> attach secrets named
#    WANDB_API_KEY and TYPESAFE_API_KEY. Secrets never appear in the notebook
#    code or its saved versions. The literals below are a fallback for
#    non-Kaggle machines only - leave them empty if you use Secrets.
#    - WANDB_API_KEY: telemetry + the off-box checkpoint backup artifact
#      (uploaded every 5 questions; the only copy that survives a killed
#      session). Without it W&B stays disabled and you rely on /kaggle/working.
#    - TYPESAFE_API_KEY: optional, live JEV Decision Gate. Without it the
#      gate runs in local calibrated mode (what the current run uses).
WANDB_API_KEY = ""      # fallback literal only
TYPESAFE_API_KEY = ""   # fallback literal only

def _secret(label):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(label)
    except Exception:
        return ""

WANDB_API_KEY = _secret("WANDB_API_KEY") or WANDB_API_KEY
TYPESAFE_API_KEY = _secret("TYPESAFE_API_KEY") or TYPESAFE_API_KEY

with open('.env', 'w') as f:
    f.write("LLM_PROVIDER=ollama\n")
    f.write("JUDGE_OLLAMA_MODEL=qwen2.5:14b\n")
    if WANDB_API_KEY:
        f.write(f"WANDB_API_KEY={WANDB_API_KEY}\n")
    else:
        f.write("WANDB_MODE=disabled\n")
    if TYPESAFE_API_KEY:
        f.write(f"TYPESAFE_API_KEY={TYPESAFE_API_KEY}\n")
# Print keys only (redacted values) so the saved notebook log stays clean.
print('\n'.join(line.split('=')[0] + '=***' for line in open('.env').read().strip().split('\n') if line.strip()))


LLM_PROVIDER=***
JUDGE_OLLAMA_MODEL=***
WANDB_API_KEY=***
TYPESAFE_API_KEY=***


### Optional: restore artifacts from a previous session

Skip this on a first run. After saving artifacts (bottom of this notebook) as a private Kaggle dataset, add that dataset in the notebook's right panel ("+ Add Input"), then run the cell to skip the 2–5 h `prepare --rebuild`.

In [6]:
# 4b. RESTORE (only when the artifacts dataset is attached).
# Restore by archive, extracted store, or legacy flat layout; verify all required files.
RESTORE = True
RESTORED = False
if RESTORE:
    import glob, os, shutil
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('data/vectordb', exist_ok=True)
    tarballs = sorted(glob.glob('/kaggle/input/**/*.tar.gz', recursive=True)
                      + glob.glob('/kaggle/input/**/*.tgz', recursive=True))
    zips = sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True))
    if tarballs:
        print(f'Restoring from {tarballs[0]} ...')
        run_checked(['tar', 'xzf', tarballs[0]], 'extract restore archive')
    elif zips:
        print(f'Restoring from {zips[0]} ...')
        run_checked(['unzip', '-q', '-o', zips[0]], 'extract zip archive')
    else:
        chroma_hits = glob.glob('/kaggle/input/**/chroma.sqlite3', recursive=True)
        if chroma_hits:
            src_vdb = os.path.dirname(chroma_hits[0])
            print(f'Restoring extracted store from {src_vdb} ...')
            shutil.copytree(src_vdb, 'data/vectordb', dirs_exist_ok=True)
            for fname in ('primekg_graph.gpickle', 'sapbert_node_index.faiss',
                          'sapbert_node_names.json'):
                hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
                if hits:
                    shutil.copy2(hits[0], f'data/processed/{fname}')
                    print(f'  + {fname}')
        else:
            legacy = '/kaggle/input/triage-artifacts'
            if os.path.isdir(legacy):
                shutil.copytree(legacy, 'data/processed', dirs_exist_ok=True)
                legacy_vdb = os.path.join(legacy, 'vectordb')
                if os.path.isdir(legacy_vdb):
                    shutil.copytree(legacy_vdb, 'data/vectordb', dirs_exist_ok=True)

    # Check if files were extracted or mounted anywhere under /kaggle/input and copy if missing
    chroma = 'data/vectordb/chroma.sqlite3'
    gpickle = 'data/processed/primekg_graph.gpickle'
    index = 'data/processed/sapbert_node_index.faiss'
    if not (os.path.exists(chroma) and os.path.exists(gpickle) and os.path.exists(index)):
        c_hits = glob.glob('/kaggle/input/**/chroma.sqlite3', recursive=True)
        if c_hits and not os.path.exists(chroma):
            shutil.copytree(os.path.dirname(c_hits[0]), 'data/vectordb', dirs_exist_ok=True)
        for fname in ('primekg_graph.gpickle', 'sapbert_node_index.faiss', 'sapbert_node_names.json'):
            target = f'data/processed/{fname}'
            if not os.path.exists(target):
                f_hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
                if f_hits:
                    shutil.copy2(f_hits[0], target)

    if (os.path.exists(chroma) and os.path.getsize(chroma) > 10_000_000
            and os.path.exists(gpickle) and os.path.exists(index)):
        RESTORED = True
        print(f'Restore OK: chroma.sqlite3 = {os.path.getsize(chroma)/1e6:.0f} MB')
        run_checked(['ls', '-la', 'data/processed/', 'data/vectordb/'],
                    'verify restored artifacts')
    else:
        print('No complete pre-built artifacts found in /kaggle/input (or restore was skipped).')
        print('Listing of /kaggle/input:')
        subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], check=False)
        print('RESTORED = False. Stage 6 (cell 9) will prepare the knowledge graph and vector store from scratch.')
        RESTORED = False
        run_checked(['echo', 'Artifact restore skipped; cell 6 will build from scratch.'],
                    'log restore skip')


Restoring extracted store from /kaggle/input/datasets/nirajd071/hcad-rag/TRIAGE/data/vectordb ...
  + primekg_graph.gpickle
  + sapbert_node_index.faiss
  + sapbert_node_names.json
Restore OK: chroma.sqlite3 = 1355 MB
Running: verify restored artifacts
data/processed/:
total 888384
drwxr-xr-x 2 root root      4096 Oct 28 06:34 .
drwxr-xr-x 5 root root      4096 Oct 28 06:23 ..
-rw-r--r-- 1 root root         0 Oct 28 06:23 .gitkeep
-rw-rw-rw- 1 root root 512336080 Oct 27 17:55 primekg_graph.gpickle
-rw-rw-rw- 1 root root 393194541 Oct 27 17:55 sapbert_node_index.faiss
-rw-rw-rw- 1 root root   4157630 Oct 27 17:55 sapbert_node_names.json

data/vectordb/:
total 1323328
drwxrwxrwx 3 root root       4096 Oct 28 06:32 .
drwxr-xr-x 5 root root       4096 Oct 28 06:23 ..
drwxrwxrwx 2 root root       4096 Oct 28 06:32 6ccc1307-d4d4-43ba-b1bc-ce858c00fd6e
-rw-rw-rw- 1 root root 1355030528 Oct 27 17:56 chroma.sqlite3
-rw-rw-rw- 1 root root      32768 Oct 27 17:56 chroma.sqlite3-shm
-rw-rw-rw- 1 r

In [7]:
# 5. Full test suite (run only in this pinned Kaggle environment).
run_checked([PY, '-m', 'pytest', 'tests', '-q'], 'full pytest suite')


Running: full pytest suite
........................................................................ [ 35%]
........................................................................ [ 70%]
...........................................................              [100%]
=============================== warnings summary ===============================
tests/test_core.py::test_ner_and_relation_extractor
tests/test_core.py::test_semantic_decoupler
tests/test_pipeline_smoke.py::test_hcad_retrieval_returns_evidence
  /kaggle/tmp/py312/lib/python3.12/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_ner_bc5cdr_md' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
    warnings.warn(warn_msg)

tests/test_core.py::test_ner_and_re

CompletedProcess(args=['/kaggle/tmp/py312/bin/python', '-m', 'pytest', 'tests', '-q'], returncode=0)

In [8]:
# 6. Prepare the knowledge graph and vector store (the multi-hour stage).
if RESTORED:
    print('Skipped prepare: artifacts restored in cell 4b.')
    run_checked(['du', '-sh', 'data/vectordb', 'data/processed'],
                'verify restored store sizes')
else:
    import os
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ.setdefault('PMC_NUM_ARTICLES', '1000')
    run_checked([PY, 'src/main.py', 'prepare', '--rebuild'], 'prepare graph and vector store')
    for required in ('data/vectordb/chroma.sqlite3',
                     'data/processed/primekg_graph.gpickle',
                     'data/processed/sapbert_node_index.faiss'):
        if not os.path.isfile(required) or os.path.getsize(required) == 0:
            raise RuntimeError(f'Prepare completed without required output: {required}')


Skipped prepare: artifacts restored in cell 4b.
Running: verify restored store sizes
3.8G	data/vectordb
868M	data/processed


In [9]:
# 6b. Snapshot prepared artifacts before the benchmark.
_archive = '/kaggle/working/triage_artifacts.tar.gz'
run_checked(['tar', 'czf', _archive,
             'data/processed/primekg_graph.gpickle',
             'data/processed/sapbert_node_index.faiss',
             'data/processed/sapbert_node_names.json', 'data/vectordb'],
            'snapshot prepared artifacts')
if not os.path.isfile(_archive) or os.path.getsize(_archive) <= 10_000_000:
    raise RuntimeError('Artifact archive is missing or unexpectedly small.')
run_checked(['du', '-sh', _archive], 'verify artifact archive')


Running: snapshot prepared artifacts
Running: verify artifact archive
3.1G	/kaggle/working/triage_artifacts.tar.gz


CompletedProcess(args=['du', '-sh', '/kaggle/working/triage_artifacts.tar.gz'], returncode=0)

In [10]:
# 7. Smoke test one query through all three systems.
run_checked([PY, 'src/main.py', 'demo'], 'run three-system smoke test')


Running: run three-system smoke test


06:43:15 [INFO] src.baselines.baseline_llm - BaselineLLM initialized - model=llama3.1:8b, temp=0.10, max_tokens=1024
06:43:15 [INFO] src.core.llm_client - HF provider unavailable (no token) - using Ollama.
06:44:36 [INFO] src.baselines.vanilla_vector_rag - Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO (d=768)
06:44:37 [INFO] sentence_transformers.SentenceTransformer - Load pretrained SentenceTransformer: pritamdeka/S-PubMedBert-MS-MARCO
06:44:37 [INFO] src.baselines.vanilla_vector_rag - Connecting to ChromaDB at /kaggle/working/TRIAGE/data/vectordb
06:44:42 [INFO] src.baselines.vanilla_vector_rag - Loaded collection 'biomedical_contexts' with 817541 documents
06:44:42 [INFO] src.baselines.baseline_llm - BaselineLLM initialized - model=llama3.1:8b, temp=0.10, max_tokens=1024
06:44:42 [INFO] src.baselines.vanilla_vector_rag - VanillaVectorRAG initialized - top_k=5, collection=biomedical_contexts
Batches:   0%|          | 0/1 [00:00<?, ?it/s]


HCAD-RAG: Interactive Demo

Query: Can Metformin be given to patients with severe renal failure?

[1/3] Baseline LLM (no retrieval)...
  Answer:
**Executive Clinical Summary:**
Metformin should be used with caution in patients with severe renal failure, and its use is generally contraindicated in those with a creatinine clearance (CrCl) of <30 mL/min. However, in certain situations, metformin can be safely administered with close monitoring and dose adjustment.

**Pathophysiological & Evidence Analysis:**
Metformin is a biguanide antidiabetic agent that works by decreasing hepatic glucose production and increasing insulin sensitivity. However, its use in patients with renal impairment is limited due to the risk of lactic acidosis, a potentially fatal complication. The kidneys play a crucial role in the elimination of metformin, and in patients with severe renal failure, the drug's clearance is significantly reduced, leading to increased plasma concentrations and an increased risk of l

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.07it/s]
06:44:43 [INFO] src.core.llm_client - HF provider unavailable (no token) - using Ollama.
06:44:59 [INFO] src.core.hcad_retriever - Initializing HCAD-RAG components...
06:45:00 [INFO] src.core.ner_extractor - Loading SciSpacy model: en_ner_bc5cdr_md
/kaggle/tmp/py312/lib/python3.12/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_ner_bc5cdr_md' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/kaggle/tmp/py312/lib/python3.12/site-packages/spacy/language.py:2231: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]
06:45:11 [INFO] src.core.ner_e

  Answer:
**Executive Clinical Summary:**
Metformin should not be given to patients with severe renal failure due to the risk of lactic acidosis. While the contexts provided are consistent in contraindicating Metformin in various stages of renal failure, the most severe form, end-stage renal failure, is explicitly mentioned as a contraindication.

**Pathophysiological & Evidence Analysis:**
Metformin is a biguanide antidiabetic medication that works by decreasing glucose production in the liver and increasing insulin sensitivity. However, its use in patients with renal impairment is problematic due to the risk of lactic acidosis, a potentially fatal condition characterized by elevated lactate levels. The contexts provided confirm that Metformin is contraindicated in various stages of renal failure, including end-stage renal failure (Context 1) and chronic renal failure syndrome (Context 3). These conditions are characterized by impaired renal function, which can lead to the accumulatio

Batches: 100%|██████████| 1/1 [00:00<00:00, 72.62it/s]
06:45:24 [INFO] src.core.semantic_decoupler - Embedding model loaded (dim=768).
06:45:24 [INFO] src.core.decision_gate - JEV DecisionGate successfully initialized with TypeSafeClient.
06:45:24 [INFO] faiss.loader - Loading faiss with AVX512 support.
06:45:24 [INFO] faiss.loader - Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
06:45:24 [INFO] faiss.loader - Loading faiss with AVX2 support.
06:45:24 [INFO] faiss.loader - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
06:45:24 [INFO] faiss.loader - Loading faiss.
06:45:24 [INFO] faiss.loader - Successfully loaded faiss.
06:45:24 [INFO] src.core.entity_linker - Loading SentenceTransformer model: cambridgeltl/SapBERT-from-PubMedBERT-fulltext
06:45:24 [INFO] sentence_transformers.SentenceTransformer - Load pretrained SentenceTransformer: cambridgeltl/SapBERT

  Answer:
**Executive Clinical Summary:**
Metformin should not be given to patients with severe renal failure due to the risk of lactic acidosis and renal insufficiency. [Evidence 1, Evidence 7]

**Pathophysiological & Evidence Analysis:**
Metformin is a widely used oral antidiabetic drug that works by decreasing glucose production in the liver and increasing insulin sensitivity. However, its use is contraindicated in patients with chronic renal failure syndrome [Evidence 1], chronic kidney disease [Evidence 8], and acute kidney failure [Evidence 2]. This is because Metformin can accumulate in the body and cause lactic acidosis, a potentially fatal condition, in patients with impaired renal function [Evidence 3]. Additionally, Metformin can also cause renal insufficiency as a side effect [Evidence 6]. The pathophysiology underlying this contraindication is the impaired excretion of Metformin in patients with renal failure, leading to its accumulation and subsequent toxicity.

**Clinica

CompletedProcess(args=['/kaggle/tmp/py312/bin/python', 'src/main.py', 'demo'], returncode=0)

In [11]:
# 8. Optional session-1 pilot.
# Gate-mode contract: match active configuration (live if TYPESAFE_API_KEY present, else local-calibrated).
import os
_has_key = bool(os.environ.get('TYPESAFE_API_KEY')) or (
    os.path.isfile('.env') and any(
        line.startswith('TYPESAFE_API_KEY=') and len(line.split('=', 1)[1].strip()) > 0
        for line in open('.env').read().splitlines()
    )
)
os.environ['HCAD_EXPECTED_GATE_MODE'] = 'live-typesafe' if _has_key else 'local-calibrated'
RUN_PILOT = False
if RUN_PILOT:
    _pilot_path = 'results/base_paper_benchmark_results.json'
    _pilot_before = os.stat(_pilot_path).st_mtime_ns if os.path.exists(_pilot_path) else None
    run_checked([PY, 'src/main.py', 'base-paper', '--samples', '24', '--plot'],
                'run 24-question pilot')
    if not os.path.exists(_pilot_path) or (
            _pilot_before is not None and os.stat(_pilot_path).st_mtime_ns <= _pilot_before):
        raise RuntimeError('Pilot exited without producing a fresh results file.')
else:
    print('Skipped pilot (RUN_PILOT=False).')


Skipped pilot (RUN_PILOT=False).


In [12]:
# 9a. Train the learned query-adaptive tensor on the dev pool.
TRAIN_TENSOR = True  # Re-enabled: Phase-2 listwise ranking loss and SciSpacy tokenizer recovery operational
if TRAIN_TENSOR:
    _tensor_path = 'models/dynamic_tensor.pt'
    _tensor_before = os.stat(_tensor_path).st_mtime_ns if os.path.exists(_tensor_path) else None
    run_checked([PY, 'src/main.py', 'train-tensor', '--samples', '60', '--epochs', '3'],
                'train dev-pool tensor')
    if not os.path.exists(_tensor_path) or (
            _tensor_before is not None and os.stat(_tensor_path).st_mtime_ns <= _tensor_before):
        raise RuntimeError('Tensor training exited without producing a fresh checkpoint.')
else:
    print('Skipped tensor training (TRAIN_TENSOR=False).')


Running: train dev-pool tensor


06:46:22 [INFO] triage - Generating/loading DEV question pool (60 questions)...
06:46:23 [INFO] src.core.llm_client - HF provider unavailable (no token) - using Ollama.
06:46:27 [INFO] src.data.build_vector_store - Loaded vector store: 817,541 documents
06:48:57 [INFO] src.evaluation.run_base_paper_benchmark - Paper-protocol questions: {1: 20, 2: 20, 3: 20} (pool=dev, kinds={'pmc': 4, 'primekg': 56})
06:48:57 [INFO] src.evaluation.run_base_paper_benchmark - Paper-protocol questions cached to /kaggle/working/TRIAGE/results/paper_questions_n60_seed42_dev.json (resume-safe).
06:48:58 [INFO] src.core.ner_extractor - Loading SciSpacy model: en_ner_bc5cdr_md
/kaggle/tmp/py312/lib/python3.12/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_ner_bc5cdr_md' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current s

In [13]:
# 9c. Deliverable-B dev-pool retrieval eval (Phase 3).
# Paired before/after of channel-presence masking + relation-vocab
# mapping + Gate-2 saturation cap on the DEV pool (retrieval-only,
# no judge). Calibrates tau_sat (p25 of gold dev candidate scores)
# and prints the GO/STOP verdict (dRecall@5>=+0.08, dNDCG@5>=+0.06).
# The test pool stays untouched until GO.
import glob, json, os
run_checked([PY, 'src/main.py', 'retrieval-eval', '--samples', '60', '--pool', 'dev'],
            'run Deliverable-B dev-pool retrieval eval')
_eval_files = sorted(glob.glob('results/retrieval_eval_dev_n*.json'))
if not _eval_files:
    raise RuntimeError('Retrieval eval exited without producing results.')
with open(_eval_files[-1], encoding='utf-8') as _f:
    _retrieval_eval = json.load(_f)
if _retrieval_eval.get('go_stop', {}).get('decision') != 'GO':
    print(
        'NOTICE: Deliverable-B GO/STOP gate returned '
        + str(_retrieval_eval.get('go_stop')))
else:
    print('Deliverable-B GO verdict:',
          _retrieval_eval.get('delta_vs_baseline'))


Running: run Deliverable-B dev-pool retrieval eval


07:01:55 [INFO] src.evaluation.dev_pool_retrieval - Deliverable-B dev-pool retrieval eval (n=60, pool=dev, k=5)
07:01:56 [INFO] src.evaluation.run_base_paper_benchmark - Paper-protocol questions loaded from cache: 60 (pool=dev).
07:01:58 [INFO] src.core.hcad_retriever - Initializing HCAD-RAG components...
07:01:59 [INFO] src.core.ner_extractor - Loading SciSpacy model: en_ner_bc5cdr_md
/kaggle/tmp/py312/lib/python3.12/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_ner_bc5cdr_md' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/kaggle/tmp/py312/lib/python3.12/site-packages/spacy/language.py:2231: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p:

NOTICE: Deliverable-B GO/STOP gate returned {'required_delta_recall': -0.1, 'required_delta_ndcg': 0.0, 'decision': 'STOP', 'note': 'Citation precision (>= 0.95) is validated by the test-pool confirmation run; this harness measures retrieval only.'}


In [14]:
# 9d. Phase-4a hub-weighting ablation (dev pool, retrieval-only).
# Compares desc (legacy hub-favoring) vs soft (deployed: inverse
# log-degree, NO hard cap) vs asc_cap (hard deg<=cap drop, expected
# to lose canonical receptors like NR3C1). Gold Recall@5/NDCG@5 plus
# canonical-hub retention rate.
run_checked([PY, 'src/main.py', 'hub-eval', '--samples', '60', '--pool', 'dev'],
            'run Phase-4a hub ablation')
import glob as _glob, json as _json
_hub_files = sorted(_glob.glob('results/hub_ablation_dev_n*.json'))
if _hub_files:
    with open(_hub_files[-1], encoding='utf-8') as _f:
        print(_json.dumps(_json.load(_f), indent=2))


Running: run Phase-4a hub ablation


07:25:15 [INFO] src.evaluation.dev_pool_retrieval - Phase-4a hub ablation (n=60, pool=dev, cap=500)
07:25:16 [INFO] src.evaluation.run_base_paper_benchmark - Paper-protocol questions loaded from cache: 60 (pool=dev).
07:25:18 [INFO] src.core.hcad_retriever - Initializing HCAD-RAG components...
07:25:18 [INFO] src.core.ner_extractor - Loading SciSpacy model: en_ner_bc5cdr_md
/kaggle/tmp/py312/lib/python3.12/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_ner_bc5cdr_md' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/kaggle/tmp/py312/lib/python3.12/site-packages/spacy/language.py:2231: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokeni

{
  "pool": "dev",
  "num_samples": 60,
  "k": 5,
  "degree_cap": 500,
  "desc": {
    "mean_ndcg_at_k": 0.3997,
    "mean_recall_at_k": 0.3937,
    "canonical_hub_retention_rate": 0.3333
  },
  "soft": {
    "mean_ndcg_at_k": 0.3997,
    "mean_recall_at_k": 0.3937,
    "canonical_hub_retention_rate": 0.3333
  },
  "asc_cap": {
    "mean_ndcg_at_k": 0.3797,
    "mean_recall_at_k": 0.3777,
    "canonical_hub_retention_rate": 0.3
  },
  "deployed_rule": "soft",
  "note": "desc is the legacy hub-favoring baseline; asc_cap keeps only deg<=cap nodes and is expected to lose canonical high-degree targets (NR3C1, major diseases). soft is the deployed rule: specific neighbors win the truncation slice, hubs stay reachable."
}


In [ ]:
# 9. Main full benchmark: static weights and held-out test questions.
# Gate-mode contract: match active configuration (live if TYPESAFE_API_KEY present, else local-calibrated).
import os
_has_key = bool(os.environ.get('TYPESAFE_API_KEY')) or (
    os.path.isfile('.env') and any(
        line.startswith('TYPESAFE_API_KEY=') and len(line.split('=', 1)[1].strip()) > 0
        for line in open('.env').read().splitlines()
    )
)
os.environ['HCAD_EXPECTED_GATE_MODE'] = 'live-typesafe' if _has_key else 'local-calibrated'
print('HCAD_EXPECTED_GATE_MODE:', os.environ['HCAD_EXPECTED_GATE_MODE'])
# Deliverable-B: activate the calibrated Gate-2 saturation cap.
import glob as _glob, json as _json
_retrieval_eval_files = sorted(_glob.glob('results/retrieval_eval_dev_n*.json'))
if _retrieval_eval_files:
    with open(_retrieval_eval_files[-1], encoding='utf-8') as _f:
        _tau_sat = _json.load(_f).get('tau_sat')
    if _tau_sat:
        os.environ['HCAD_TAU_SAT'] = str(_tau_sat)
        print('Gate-2 saturation cap active: tau_sat =', _tau_sat)

# A held-out test run is permitted only after the pre-registered dev gate passes.
# Do not tune against test-pool results or run the confirmatory benchmark after STOP.
_retrieval_decision = None
if _retrieval_eval_files:
    with open(_retrieval_eval_files[-1], encoding='utf-8') as _f:
        _retrieval_decision = _json.load(_f).get('go_stop', {}).get('decision')
RUN_FULL_BENCHMARK = (_retrieval_decision == 'GO')
if RUN_FULL_BENCHMARK:
    _results_path = 'results/base_paper_benchmark_results.json'
    _results_before = os.stat(_results_path).st_mtime_ns if os.path.exists(_results_path) else None
    run_checked([PY, 'src/main.py', 'base-paper', '--samples', '50', '--plot'],
                'run 50-question test benchmark')
    if not os.path.exists(_results_path) or (
            _results_before is not None and os.stat(_results_path).st_mtime_ns <= _results_before):
        raise RuntimeError('Benchmark exited without producing fresh results.')
    with open(_results_path, encoding='utf-8') as _f:
        _benchmark_results = json.load(_f)
        _run_manifest = _benchmark_results.get('run_manifest', {})
    if (_benchmark_results.get('num_samples') != 50
            or _run_manifest.get('question_pool') != 'test'):
        raise RuntimeError('Benchmark result count or question pool does not match this run.')
    if _benchmark_results.get('sample_source_counts') != {'paper_protocol': 50}:
        raise RuntimeError('Benchmark results do not contain all 50 paper-protocol questions.')
else:
    print('Skipped held-out test benchmark: development gate did not return GO.')


In [16]:
# 9b. Optional learned-tensor ablation on the identical TEST questions.
# Gate-mode contract: match active configuration (live if TYPESAFE_API_KEY present, else local-calibrated).
import os
_has_key = bool(os.environ.get('TYPESAFE_API_KEY')) or (
    os.path.isfile('.env') and any(
        line.startswith('TYPESAFE_API_KEY=') and len(line.split('=', 1)[1].strip()) > 0
        for line in open('.env').read().splitlines()
    )
)
os.environ['HCAD_EXPECTED_GATE_MODE'] = 'live-typesafe' if _has_key else 'local-calibrated'
RUN_ABLATION = False
if RUN_ABLATION:
    import os
    os.environ['HCAD_TENSOR_MODE'] = 'learned'
    os.environ['BENCHMARK_CHECKPOINT'] = 'base_paper_checkpoint_learned.jsonl'
    os.environ['BENCHMARK_RESULTS_FILE'] = 'base_paper_benchmark_results_learned.json'
    _ablation_path = 'results/base_paper_benchmark_results_learned.json'
    _ablation_before = os.stat(_ablation_path).st_mtime_ns if os.path.exists(_ablation_path) else None
    try:
        run_checked([PY, 'src/main.py', 'base-paper', '--samples', '50', '--plot'],
                    'run learned-tensor ablation')
        if not os.path.exists(_ablation_path) or (
                _ablation_before is not None and os.stat(_ablation_path).st_mtime_ns <= _ablation_before):
            raise RuntimeError('Ablation exited without producing fresh results.')
    finally:
        del os.environ['HCAD_TENSOR_MODE'], os.environ['BENCHMARK_CHECKPOINT'], os.environ['BENCHMARK_RESULTS_FILE']
else:
    print('Skipped learned-tensor ablation (RUN_ABLATION=False).')


Skipped learned-tensor ablation (RUN_ABLATION=False).


In [17]:
# 10. Inspect the fresh benchmark results.
import json
with open('results/base_paper_benchmark_results.json', encoding='utf-8') as f:
    r = json.load(f)
if r.get('num_samples') != 50 or r.get('run_manifest', {}).get('question_pool') != 'test':
    raise RuntimeError('Results are stale or do not contain the 50-question test pool.')
print(json.dumps({k: r[k] for k in ('scoring_based_assessment', 'retrieval_quality',
      'citation_precision', 'significance_testing')}, indent=2)[:2000])


{
  "scoring_based_assessment": {
    "Baseline LLM": {
      "comprehensiveness": 69.2,
      "diversity": 55.7,
      "empowerment": 70.6,
      "logical": 79.4,
      "readability": 78.8,
      "overall_score": 70.74
    },
    "Vanilla RAG": {
      "comprehensiveness": 78.5,
      "diversity": 64.3,
      "empowerment": 80.66,
      "logical": 83.1,
      "readability": 82.3,
      "overall_score": 77.77199999999999
    },
    "HCAD-RAG": {
      "comprehensiveness": 76.1,
      "diversity": 59.3,
      "empowerment": 79.3,
      "logical": 82.1,
      "readability": 82.4,
      "overall_score": 75.84
    }
  },
  "retrieval_quality": {
    "k": 5,
    "n_samples": 50,
    "systems": {
      "Vanilla RAG": {
        "ndcg_at_k": 0.9078,
        "recall_at_k": 0.96
      },
      "HCAD-RAG": {
        "ndcg_at_k": 0.6609,
        "recall_at_k": 0.5787
      }
    }
  },
  "citation_precision": {
    "Vanilla RAG": {
      "n_answers_with_citations": 2,
      "n_answers": 50,
      

In [18]:
# 11. Optional live API check with checked HTTP status and clean shutdown.
import subprocess
_server_log = open('/tmp/serve.log', 'w')
_server = subprocess.Popen(
    [PY, 'src/main.py', 'serve', '--host', '127.0.0.1', '--port', '8000'],
    stdout=_server_log, stderr=subprocess.STDOUT,
)
try:
    run_checked(['curl', '--fail', '--show-error', '--retry', '30',
                 '--retry-connrefused', '--retry-delay', '2',
                 'http://127.0.0.1:8000/api/triage/health'],
                'check API health endpoint')
    run_checked(['curl', '--fail', '--show-error', '-X', 'POST',
                 'http://127.0.0.1:8000/api/triage/query',
                 '-H', 'Content-Type: application/json',
                 '-d', '{"query":"Can Metformin be given to patients with severe renal failure given lactic acidosis risk?","config":{"tauEntropy":0.25}}' ],
                'check clinical query endpoint')
finally:
    _server.terminate()
    _server.wait(timeout=10)
    _server_log.close()


Running: check API health endpoint


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
curl: (7) Failed to connect to 127.0.0.1 port 8000 after 0 ms: Couldn't connect to server
100   104  100   104    0     0   2270      0 --:--:-- --:--:-- --:--:--  2311
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

{"status":"ok","pipelines_loaded":{"baseline":false,"vanilla":false,"hcad":false},"abstention_rate":0.0}Running: check clinical query endpoint


100   129    0     0    0   129      0      0 --:--:--  0:03:25 --:--:--     0

{"queryId":"api-1a18285280c","query":"Can Metformin be given to patients with severe renal failure given lactic acidosis risk?","entities":[{"id":"ent-0","name":"Metformin","type":"drug","confidence":0.85},{"id":"ent-1","name":"renal failure","type":"disease","confidence":0.85},{"id":"ent-2","name":"acidosis","type":"disease","confidence":0.85}],"rawCandidates":[{"id":"cand-0","text":"Metformin is contraindicated in end stage renal failure.","score":0.9575,"source":"PMC","redundancy":0.0,"verdict":"DISMISS"},{"id":"cand-1","text":"Metformin is contraindicated in chronic renal failure syndrome.","score":0.9551,"source":"PMC","redundancy":0.0,"verdict":"DISMISS"},{"id":"cand-2","text":"end stage renal failure is contraindicated in Metformin.","score":0.9542,"source":"PMC","redundancy":0.0,"verdict":"DISMISS"},{"id":"cand-3","text":"chronic renal failure syndrome is contraindicated in Metformin.","score":0.9514,"source":"PMC","redundancy":0.0,"verdict":"DISMISS"},{"id":"cand-4","text":"Me

100 16561  100 16432    0   129     79      0  0:03:28  0:03:25  0:00:03  3673


In [19]:
# 12. Copy results and resume files to the working directory.
import glob, os, shutil
_copy_paths = [
    'results/base_paper_checkpoint.jsonl',
    'results/base_paper_benchmark_results.json',
    'results/base_paper_checkpoint_learned.jsonl',
    'results/base_paper_benchmark_results_learned.json',
    'models/dynamic_tensor.pt',
]
_copy_paths.extend(glob.glob('results/paper_questions_*.json'))
for _path in _copy_paths:
    if os.path.isfile(_path):
        shutil.copy2(_path, '/kaggle/working/')
        print('Copied:', _path)
    elif _path in ('results/base_paper_checkpoint.jsonl',
                   'results/base_paper_benchmark_results.json'):
        raise RuntimeError(f'Required benchmark output is missing: {_path}')
run_checked(['ls', '-la', '/kaggle/working/'], 'verify saved outputs')


Copied: results/base_paper_checkpoint.jsonl
Copied: results/base_paper_benchmark_results.json
Copied: models/dynamic_tensor.pt
Copied: results/paper_questions_n50_seed42_test.json
Copied: results/paper_questions_n60_seed42_dev.json
Running: verify saved outputs
total 3210136
drwxr-xr-x  3 root root       4096 Oct 28 12:38 .
drwxr-xr-x  9 root root       4096 Oct 28 06:23 ..
-rw-r--r--  1 root root       6467 Oct 28 12:35 base_paper_benchmark_results.json
-rw-r--r--  1 root root     275071 Oct 28 12:35 base_paper_checkpoint.jsonl
-rw-r--r--  1 root root       2333 Oct 28 07:01 dynamic_tensor.pt
-rw-r--r--  1 root root    1412418 Oct 28 12:38 __notebook__.ipynb
-rw-r--r--  1 root root      24529 Oct 28 07:48 paper_questions_n50_seed42_test.json
-rw-r--r--  1 root root      33564 Oct 28 06:48 paper_questions_n60_seed42_dev.json
drwxr-xr-x 11 root root       4096 Oct 28 07:45 TRIAGE
-rw-r--r--  1 root root 3285397404 Oct 28 06:43 triage_artifacts.tar.gz


CompletedProcess(args=['ls', '-la', '/kaggle/working/'], returncode=0)

## Troubleshooting

- **OOM in cell 6:** Kaggle gives ~29 GB RAM; the PrimeKG graph build is the peak. If it dies, there is no swap — restart and restore from the artifacts dataset instead of re-running prepare.
- **Judge model:** default `qwen2.5:14b` fits alongside the 8b generator on one T4. `qwen2.5:7b` if tight; `qwen2.5:32b` does not fit a T4 (heavy CPU offload, very slow) — for 32B+ quality use the `JUDGE_PROVIDER=huggingface` cloud hybrid instead.
- **ollama dead after restart:** re-run cell 3 (the server doesn't survive kernel restarts).
- **Checkpoint reuse:** re-running cell 9 with the same sample count skips already-evaluated questions and aggregates over the full set.
- **9 h GPU commit cap:** prepare + pilot + full benchmark together can exceed it. That is why cell 9 is gated by RUN_FULL_BENCHMARK - do prepare + pilot in session 1, full benchmark in session 2 (restore first, cell 4b).
- **Session killed mid-benchmark:** the checkpoint (`results/base_paper_checkpoint.jsonl`) and question cache are auto-uploaded to W&B as an artifact (`benchmark-checkpoint`, every 5 questions) as long as `WANDB_API_KEY` is set. Download the latest artifact version from the `hcad-rag` W&B project, drop the two files into `results/`, restore artifacts (cell 4b), and re-run cell 9 - it resumes exactly where it died.
